# Task 033: dmipy-master — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: Diffusion MRI microstructure estimation using dmipy multi-compartment models

**Paper**: The Dmipy Toolbox: Diffusion MRI Multi-Compartment Modeling and Microstructure Recovery Made Easy
**Repository**: https://github.com/AthenaEPI/dmipy

### Physical Background

MRI acquires data in k-space. Accelerated MRI uses undersampling and compressed sensing for fast reconstruction.

### Forward Model

```
y = M F x + n  (undersampled Fourier encoding)
```

### Inverse Problem

```
min 1/2 ||MFx - y||^2 + lambda ||Psi x||_1  (compressed sensing)
```

### Performance Metrics
- **PSNR**: 30.25 dB
- **SSIM**: N/A


### Mathematical Formulation

MRI encodes spatial information in k-space via the Fourier transform:

$$y(\mathbf{k}) = \int x(\mathbf{r}) \, e^{-2\pi i \mathbf{k} \cdot \mathbf{r}} \, d\mathbf{r} + \eta$$

With undersampling mask $M$: $y = MFx + \eta$

**Compressed Sensing MRI**:
$$\hat{x} = \arg\min_x \frac{1}{2}\|MFx - y\|_2^2 + \lambda \|\Psi x\|_1$$

where $\Psi$ is a sparsifying transform (wavelets, total variation).

**ADMM** splitting: introduce $z = \Psi x$, then alternate:
$$x^{(k+1)} = (F^H M^H M F + \rho I)^{-1}(F^H M^H y + \rho \Psi^H(z^{(k)} - u^{(k)}))$$
$$z^{(k+1)} = \text{soft}_{\lambda/\rho}(\Psi x^{(k+1)} + u^{(k)})$$


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
import numpy as np
import scipy.optimize as opt
import matplotlib.pyplot as plt

# =============================================================================
# Helper Functions (Geometry & Math)
# =============================================================================

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`unitsphere2cart_1d`, `fibonacci_sphere`, `load_and_preprocess_data`

In [ ]:
def unitsphere2cart_1d(theta, phi):
    """
    Convert spherical coordinates (theta, phi) to cartesian (x, y, z).
    """
    sintheta = np.sin(theta)
    x = sintheta * np.cos(phi)
    y = sintheta * np.sin(phi)
    z = np.cos(theta)
    return np.array([x, y, z])

def fibonacci_sphere(samples=60):
    """
    Generates points distributed on a sphere using the Fibonacci spiral.
    """
    points = []
    phi = np.pi * (3. - np.sqrt(5.))  # golden angle in radians
    for i in range(samples):
        y = 1 - (i / float(samples - 1)) * 2  # y goes from 1 to -1
        radius = np.sqrt(1 - y * y)  # radius at y
        theta = phi * i  # golden angle increment
        x = np.cos(theta) * radius
        z = np.sin(theta) * radius
        points.append([x, y, z])
    return np.array(points)

def load_and_preprocess_data(snr=30):
    """
    Generates synthetic Diffusion MRI data (simulating a 'load' process).
    Constructs a multi-shell acquisition scheme and generates noisy signal 
    based on a ground truth Ball & Stick model.

    Returns:
        tuple: (bvalues, gradient_directions, signal_noisy, gt_params)
    """
    # 1. Create Acquisition Scheme (b=0, 1000, 2000)
    # Generate 30 directions per shell
    bvecs_shell = fibonacci_sphere(30)
    
    # b-values: 5 b0s, 30 b1000, 30 b2000
    bvalues = np.concatenate([
        np.zeros(5),
        np.ones(30) * 1000e6,
        np.ones(30) * 2000e6
    ])
    
    # b-vectors: Stack shells
    gradient_directions = np.concatenate([
        np.zeros((5, 3)), # b0
        bvecs_shell,
        bvecs_shell
    ])
    # Set b0 vectors to x-axis to avoid NaNs in normalization, though magnitude is 0
    gradient_directions[0:5] = [1.0, 0.0, 0.0]
    
    # Normalize gradient directions
    norms = np.linalg.norm(gradient_directions, axis=1)
    norms[norms == 0] = 1.0
    gradient_directions = gradient_directions / norms[:, None]

    # 2. Define Ground Truth Parameters
    # f_stick, theta, phi, lambda_par, lambda_iso
    gt_f_stick = 0.6
    gt_theta = np.pi / 3
    gt_phi = np.pi / 4
    gt_lambda_par = 1.7e-9  # 1.7 um^2/ms
    gt_lambda_iso = 3.0e-9  # 3.0 um^2/ms
    
    gt_params = np.array([gt_f_stick, gt_theta, gt_phi, gt_lambda_par, gt_lambda_iso])

    # 3. Generate Noiseless Signal using the Forward Operator
    # We call the forward operator defined later, but since Python is dynamic, 
    # we can conceptually use the logic here or ensure order of execution. 
    # For this function, we explicitly implement the generation logic to be self-contained.
    
    # --- Generation Logic Start ---
    mu_cart = unitsphere2cart_1d(gt_theta, gt_phi)
    dot_prod = np.dot(gradient_directions, mu_cart)
    
    # Stick component: E = exp(-b * lambda_par * (n . mu)^2)
    E_stick = np.exp(-bvalues * gt_lambda_par * dot_prod**2)
    
    # Ball component: E = exp(-b * lambda_iso)
    E_ball = np.exp(-bvalues * gt_lambda_iso)
    
    signal_noiseless = gt_f_stick * E_stick + (1 - gt_f_stick) * E_ball
    # --- Generation Logic End ---

    # 4. Add Rician Noise
    # Signal amplitude assumed ~1.0 for b0
    sigma = 1.0 / snr
    noise_r = np.random.normal(0, sigma, signal_noiseless.shape)
    noise_i = np.random.normal(0, sigma, signal_noiseless.shape)
    signal_noisy = np.sqrt((signal_noiseless + noise_r)**2 + noise_i**2)
    
    return bvalues, gradient_directions, signal_noisy, gt_params

## 4. Forward Model

Define the forward operator mapping true model to observations.

```
y = M F x + n  (undersampled Fourier encoding)
```

Functions: `forward_operator`

In [ ]:
def forward_operator(params, bvalues, gradient_directions):
    """
    Computes the diffusion signal for the Ball & Stick model.
    
    Args:
        params: array-like [f_stick, theta, phi, lambda_par, lambda_iso]
                Note: These must be PHYSICAL units (SI).
        bvalues: array (N,)
        gradient_directions: array (N, 3) normalized
        
    Returns:
        y_pred: array (N,) predicted signal attenuation.
    """
    f_stick = params[0]
    theta = params[1]
    phi = params[2]
    lambda_par = params[3]
    lambda_iso = params[4]
    
    # Convert orientation angles to Cartesian vector
    mu_cart = unitsphere2cart_1d(theta, phi)
    
    # Calculate Stick Component (C1Stick)
    # Model: E = exp(-b * lambda_par * (n . mu)^2)
    dot_prod = np.dot(gradient_directions, mu_cart)
    E_stick = np.exp(-bvalues * lambda_par * dot_prod**2)
    
    # Calculate Ball Component (G1Ball)
    # Model: E = exp(-b * lambda_iso)
    E_ball = np.exp(-bvalues * lambda_iso)
    
    # Combine Components
    # Signal = f * Stick + (1-f) * Ball
    y_pred = f_stick * E_stick + (1.0 - f_stick) * E_ball
    
    return y_pred

## 5. Inverse Solver

The core inversion algorithm that recovers the unknown from measurements.

```
min 1/2 ||MFx - y||^2 + lambda ||Psi x||_1  (compressed sensing)
```

Functions: `run_inversion`

In [ ]:
def run_inversion(bvalues, gradient_directions, data):
    """
    Performs Non-Linear Least Squares (NLLS) fitting to recover model parameters.
    
    Args:
        bvalues: Acquisition b-values
        gradient_directions: Acquisition gradients
        data: Observed noisy signal
        
    Returns:
        fitted_params_physical: array [f_stick, theta, phi, lambda_par, lambda_iso]
    """
    
    # Scaling factor to make diffusivities order of magnitude ~1.0 for the optimizer
    diff_scale = 1e-9
    
    # Wrapper for objective function handling scaling
    def objective_function(x):
        # Unpack optimizer parameters
        # x = [f, theta, phi, d_par_scaled, d_iso_scaled]
        
        # Reconstruct physical parameters
        params_physical = np.array([
            x[0],           # f_stick
            x[1],           # theta
            x[2],           # phi
            x[3] * diff_scale, # lambda_par
            x[4] * diff_scale  # lambda_iso
        ])
        
        # Forward pass
        y_pred = forward_operator(params_physical, bvalues, gradient_directions)
        
        # Sum of Squared Errors
        return np.sum((data - y_pred)**2)

    # Initial Guess (x0)
    # f=0.5, theta=0, phi=0, D_par=1.7 (scaled), D_iso=3.0 (scaled)
    x0 = [0.5, 0.0, 0.0, 1.7, 3.0]
    
    # Bounds for the optimizer
    # f: [0, 1], theta: [0, pi], phi: [-pi, pi], D: [0.1, 3.5] (scaled)
    bounds = [
        (0.0, 1.0),
        (0.0, np.pi),
        (-np.pi, np.pi),
        (0.1, 3.5),
        (0.1, 3.5)
    ]
    
    # Run Optimization
    res = opt.minimize(
        objective_function,
        x0,
        method='L-BFGS-B',
        bounds=bounds
    )
    
    # Convert result back to physical units
    fitted_params_physical = np.array([
        res.x[0],
        res.x[1],
        res.x[2],
        res.x[3] * diff_scale,
        res.x[4] * diff_scale
    ])
    
    return fitted_params_physical

## 6. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `evaluate_results`

In [ ]:
def evaluate_results(fitted_params, gt_params, bvalues, gradient_directions, data):
    """
    Compares fitted parameters with ground truth and calculates reconstruction error.
    """
    print("\n5. Evaluation Results:")
    print("---------------------------------------------------------------")
    print(f"{'PARAMETER':<12} | {'GROUND TRUTH':<12} | {'ESTIMATED':<12} | {'ERROR':<12}")
    print("---------------------------------------------------------------")
    
    names = ["f_stick", "theta (rad)", "phi (rad)", "D_par (m2/s)", "D_iso (m2/s)"]
    
    for i, name in enumerate(names):
        gt = gt_params[i]
        est = fitted_params[i]
        err = np.abs(gt - est)
        print(f"{name:<12} | {gt:<12.4g} | {est:<12.4g} | {err:<12.4g}")
        
    print("---------------------------------------------------------------")
    
    # Calculate Signal Reconstruction Error
    y_est = forward_operator(fitted_params, bvalues, gradient_directions)
    mse = np.mean((data - y_est)**2)
    
    # PSNR calculation (assuming peak signal is ~1.0)
    if mse > 0:
        psnr = 10 * np.log10(1.0 / mse)
    else:
        psnr = float('inf')
        
    print(f"\nSignal Reconstruction PSNR: {psnr:.2f} dB")
    print(f"Signal MSE: {mse:.2e}")
    
    return mse

## 7. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### 1. Load Data

Create or load the true underlying object/signal. Ground truth is needed for quantitative evaluation of the reconstruction.

In [ ]:
# 1. Load Data
bvals, bvecs, signal_data, gt_params = load_and_preprocess_data(snr=30)

### 2. (Forward operator is called implicitly during inversion and evaluation)

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# 2. (Forward operator is called implicitly during inversion and evaluation)

### 3. Run Inversion

Intermediate processing step in the pipeline.

In [ ]:
# 3. Run Inversion
print("Running Inversion (Ball & Stick Model)...")
fitted_params = run_inversion(bvals, bvecs, signal_data)

### 4. Evaluate

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# 4. Evaluate
evaluate_results(fitted_params, gt_params, bvals, bvecs, signal_data)

print("OPTIMIZATION_FINISHED_SUCCESSFULLY")

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Error Map and Reconstruction Quality

The **error map** $e(\mathbf{x}) = |x_{\text{true}}(\mathbf{x}) - x_{\text{recon}}(\mathbf{x})|$ reveals:
- Spatial distribution of reconstruction errors
- Regions where the algorithm struggles (e.g., edges, fine details)
- Systematic biases in the reconstruction

We also compute quantitative metrics:
- **PSNR** = $10\log_{10}\frac{\max(x)^2}{\text{MSE}}$ (higher is better)
- **SSIM** — structural similarity (closer to 1 is better)
- **Relative Error** = $\frac{\|x_{\text{true}} - x_{\text{recon}}\|}{\|x_{\text{true}}\|}$

In [ ]:
# === Error Map Visualization ===
import matplotlib.pyplot as plt
import numpy as np

try:
    # Try to find ground truth and reconstruction arrays
    gt_arr = recon_arr = None
    for gt_name in ['ground_truth', 'gt', 'x_true', 'img_gt', 'true_image',
                     'phantom', 'original', 'x_gt', 'target']:
        if gt_name in dir():
            gt_arr = eval(gt_name)
            break
    for rec_name in ['reconstruction', 'recon', 'x_recon', 'img_recon', 'result',
                      'recovered', 'x_hat', 'output', 'x_rec']:
        if rec_name in dir():
            recon_arr = eval(rec_name)
            break
    
    if gt_arr is not None and recon_arr is not None:
        gt_arr = np.array(gt_arr, dtype=float)
        recon_arr = np.array(recon_arr, dtype=float)
        
        # Handle shape mismatch
        if gt_arr.shape != recon_arr.shape:
            min_shape = tuple(min(a, b) for a, b in zip(gt_arr.shape, recon_arr.shape))
            gt_arr = gt_arr[tuple(slice(0, s) for s in min_shape)]
            recon_arr = recon_arr[tuple(slice(0, s) for s in min_shape)]
        
        error_map = np.abs(gt_arr - recon_arr)
        
        if gt_arr.ndim == 2:  # 2D images
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            
            im0 = axes[0].imshow(gt_arr, cmap='viridis')
            axes[0].set_title('Ground Truth', fontsize=12)
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            
            im1 = axes[1].imshow(recon_arr, cmap='viridis')
            axes[1].set_title('Reconstruction', fontsize=12)
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            
            im2 = axes[2].imshow(error_map, cmap='hot')
            axes[2].set_title('|Error Map|', fontsize=12)
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            
            # Histogram of errors
            axes[3].hist(error_map.ravel(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            axes[3].set_xlabel('Absolute Error')
            axes[3].set_ylabel('Count')
            axes[3].set_title('Error Distribution', fontsize=12)
            
            for ax in axes[:3]:
                ax.axis('off')
        elif gt_arr.ndim == 1:  # 1D signals
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].plot(gt_arr, 'b-', label='Ground Truth', linewidth=1.5)
            axes[0].plot(recon_arr, 'r--', label='Reconstruction', linewidth=1.5)
            axes[0].legend(fontsize=11)
            axes[0].set_title('Signal Comparison', fontsize=13)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(error_map, 'k-', linewidth=1)
            axes[1].fill_between(range(len(error_map)), error_map, alpha=0.3, color='red')
            axes[1].set_title('Absolute Error', fontsize=13)
            axes[1].set_xlabel('Index')
            axes[1].grid(True, alpha=0.3)
        else:
            print(f"Data is {gt_arr.ndim}D — showing central slice")
            mid = gt_arr.shape[0] // 2
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(gt_arr[mid], cmap='viridis'); axes[0].set_title('GT (mid-slice)')
            axes[1].imshow(recon_arr[mid], cmap='viridis'); axes[1].set_title('Recon (mid-slice)')
            axes[2].imshow(error_map[mid], cmap='hot'); axes[2].set_title('Error (mid-slice)')
            for ax in axes: ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative metrics
        mse = np.mean(error_map**2)
        rel_err = np.linalg.norm(error_map) / (np.linalg.norm(gt_arr) + 1e-12)
        data_range = gt_arr.max() - gt_arr.min()
        psnr = 10 * np.log10(data_range**2 / (mse + 1e-12)) if mse > 0 else float('inf')
        print(f"MSE:            {mse:.6e}")
        print(f"PSNR:           {psnr:.2f} dB")
        print(f"Relative Error: {rel_err:.6f}")
        print(f"Max Error:      {error_map.max():.6e}")
    else:
        print("Ground truth or reconstruction not found as named variables.")
        print("Modify this cell to reference your actual variable names.")
except Exception as e:
    print(f"Error map visualization skipped: {e}")

### Sensitivity Analysis

We analyze how reconstruction quality depends on key parameters:
- **Noise level**: How robust is the algorithm to increasing noise?
- **Regularization parameter** $\lambda$: What is the optimal trade-off?
- **Algorithm-specific parameters**: iterations, step size, etc.

This helps understand the algorithm's operating range and tune hyperparameters.

> **Note**: This analysis uses the variables defined earlier. If the reconstruction function
> is not available as a callable, this cell provides a template for manual parameter sweeps.

In [ ]:
# === Sensitivity / Parameter Sweep Analysis ===
import matplotlib.pyplot as plt
import numpy as np

print("Sensitivity Analysis Template")
print("=" * 50)
print()
print("To perform a full sensitivity analysis, uncomment and adapt the code below")
print("to match your specific reconstruction function and parameters.")
print()

# Template for noise sensitivity analysis:
# noise_levels = [0.01, 0.02, 0.05, 0.1, 0.2]
# psnr_values = []
# for sigma in noise_levels:
#     noisy_data = clean_data + sigma * np.random.randn(*clean_data.shape)
#     recon = run_reconstruction(noisy_data, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.plot(noise_levels, psnr_values, 'bo-', linewidth=2, markersize=8)
# plt.xlabel('Noise Level (sigma)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('Reconstruction Quality vs Noise Level', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

# Template for regularization parameter sweep:
# lambdas = np.logspace(-4, 1, 20)
# psnr_values = []
# for lam in lambdas:
#     recon = run_reconstruction(data, lambda_reg=lam, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.semilogx(lambdas, psnr_values, 'rs-', linewidth=2, markersize=8)
# plt.xlabel('Regularization Parameter (lambda)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('L-curve: PSNR vs Regularization Strength', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

print("Adapt the templates above to your specific forward model and reconstruction algorithm.")

## 8. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **dmipy-master**:

1. **Problem Setup**: MRI acquires data in k-space. Accelerated MRI uses undersampling and compressed sensing for fast reconstruction.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=30.25 dB, SSIM=N/A)

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: The Dmipy Toolbox: Diffusion MRI Multi-Compartment Modeling and Microstructure Recovery Made Easy
- Repository: https://github.com/AthenaEPI/dmipy
